In [1]:
from datasets import load_from_disk
import pandas as pd
import numpy as np

/Users/layvvs/Desktop/HSE/Studying/year-project/hse-ai-year-project-2025/checkpoint-5-bogdan-egor/bogdan/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
likes = load_from_disk('dataset-parts/yambda_likes')
dislikes = load_from_disk('dataset-parts/yambda_dislikes')
# listens = load_from_disk('dataset-parts/yambda_listens')

likes_table = likes.data.table
dislikes_table = dislikes.data.table
# listens_table = listens.data.table


likes_df: pd.DataFrame = pd.DataFrame.from_arrow(likes_table)
dislikes_df: pd.DataFrame = pd.DataFrame.from_arrow(dislikes_table)
# listens_df: pd.DataFrame = pd.DataFrame.from_arrow(listens_table)

likes_df['event_type'] = 'like'
dislikes_df['event_type'] = 'dislike'
# listens_df['event_type'] = 'listen'

# yambda_df = pd.concat([likes_df, dislikes_df, listens_df], ignore_index=True)
yambda_df = pd.concat([likes_df, dislikes_df], ignore_index=True)

yambda_df = yambda_df.sort_values('timestamp')

yambda_df

,uid,timestamp,item_id,is_organic,event_type
968657,794500,75,5559703,1,dislike
968664,794500,75,2224183,1,dislike
968663,794500,75,1861694,1,dislike
968662,794500,75,2605934,1,dislike
968661,794500,75,1113563,1,dislike
...,...,...,...,...,...
713215,809400,25999905,1264650,1,like
146824,168100,25999910,83561,1,like
360130,411500,25999945,1342000,1,like
933449,411500,25999945,6603749,1,dislike


Для простоты я откинул ивенты по типу unlike, undislike, listen

Также мы договорились, что пока будем пробовать только органические данные, то есть данные, где is_organic = 1

In [3]:
yambda_df_organic = yambda_df[yambda_df['is_organic'] == 1].drop(['is_organic'], axis=1)

yambda_df_organic

,uid,timestamp,item_id,event_type
968657,794500,75,5559703,dislike
968664,794500,75,2224183,dislike
968663,794500,75,1861694,dislike
968662,794500,75,2605934,dislike
968661,794500,75,1113563,dislike
...,...,...,...,...
146823,168100,25999890,4626898,like
713215,809400,25999905,1264650,like
146824,168100,25999910,83561,like
360130,411500,25999945,1342000,like


In [4]:
yambda_df_organic['uid'].nunique(), yambda_df_organic['item_id'].nunique()

(8269, 153314)

Давайте разобьем данные на тест и трейн

Таймстемпы тут начинаются относительно сбора времени начало сбора данных. Они поделены на бины по 5 секунд, то есть значение 15 - это 15 * 5, то есть 75 секунда с момента начала сбора. Самое максимальное значение = 26000000, то есть 130000000 секунд, а это ~4 года. Я хочу взять последний год для трейна и последнюю неделю для теста.

In [5]:
weak_ticks = (7 * 24 * 3600) // 5
year_ticks = (365 * 24 * 3600) // 5

weak_ticks, year_ticks

(120960, 6307200)

In [6]:
max_time = yambda_df_organic['timestamp'].max()

test_start = max_time - weak_ticks
train_start = test_start - year_ticks

In [7]:
train_df = yambda_df_organic[
    (yambda_df_organic['timestamp'] >= train_start) &
    (yambda_df_organic['timestamp'] < test_start)
]

test_df = yambda_df_organic[yambda_df_organic['timestamp'] >= test_start]

In [8]:
train_df

,uid,timestamp,item_id,event_type
983103,963800,19571810,5388750,dislike
978827,912500,19571875,7013020,dislike
810592,912500,19571875,7013020,like
810593,912500,19571975,4621681,like
140458,159000,19571990,3061013,like
...,...,...,...,...
25487,32500,25878800,5016149,like
505853,582900,25878910,3587449,like
306800,348700,25878920,3712472,like
477977,553400,25878945,6168716,like


In [9]:
test_df

,uid,timestamp,item_id,event_type
652648,753100,25879060,7260491,like
816771,922200,25879095,1265696,like
752703,849100,25879170,134761,like
196984,228000,25879265,4564073,like
196985,228000,25879275,3004976,like
...,...,...,...,...
146823,168100,25999890,4626898,like
713215,809400,25999905,1264650,like
146824,168100,25999910,83561,like
360130,411500,25999945,1342000,like


Построим матрицу взаимодействий для тренировки

In [10]:
train_df_sorted = train_df.sort_values(by=['uid', 'item_id', 'timestamp'])

final_states = train_df_sorted.drop_duplicates(subset=['uid', 'item_id'], keep='last').copy()

weight_mapping = {
    'like': 1,
    'dislike': 0,
}
final_states['interaction_weight'] = final_states['event_type'].map(weight_mapping)

user_item = final_states[['uid', 'item_id', 'interaction_weight']].reset_index(drop=True)

In [11]:
user_item

,uid,item_id,interaction_weight
0,100,2263048,1
1,100,3526521,1
2,100,4927727,0
3,100,5590855,1
4,300,1126058,1
...,...,...,...
177400,999900,1369814,1
177401,999900,1524105,1
177402,999900,2185191,1
177403,999900,2467371,1


Удалим из теста тех юзеров и айтемы, которых нет в трейне

In [12]:
train_users = set(user_item['uid'])
train_items = set(user_item['item_id'])

test_clean = test_df[
    test_df['uid'].isin(train_users) &
    test_df['item_id'].isin(train_items)
]

In [13]:
user_item

,uid,item_id,interaction_weight
0,100,2263048,1
1,100,3526521,1
2,100,4927727,0
3,100,5590855,1
4,300,1126058,1
...,...,...,...
177400,999900,1369814,1
177401,999900,1524105,1
177402,999900,2185191,1
177403,999900,2467371,1


In [14]:
test_clean

,uid,timestamp,item_id,event_type
816771,922200,25879095,1265696,like
752703,849100,25879170,134761,like
196984,228000,25879265,4564073,like
196985,228000,25879275,3004976,like
642226,741400,25879385,426891,like
...,...,...,...,...
443959,515100,25998830,6314715,like
597381,691200,25998930,3691559,like
184952,215400,25999505,6168716,like
699924,796000,25999610,5368606,like


In [15]:
item2id = {k: v for v, k in enumerate(user_item['item_id'].unique())}
user2id = {k: v for v, k in enumerate(user_item['uid'].unique())}

id2item = {v: k for k, v in item2id.items()}
id2user = {v: k for k, v in user2id.items()}

In [16]:
user_item['uidx'] = user_item['uid'].map(user2id)
user_item['item_idx'] = user_item['item_id'].map(item2id)

In [17]:
user_item

,uid,item_id,interaction_weight,uidx,item_idx
0,100,2263048,1,0,0
1,100,3526521,1,0,1
2,100,4927727,0,0,2
3,100,5590855,1,0,3
4,300,1126058,1,1,4
...,...,...,...,...,...
177400,999900,1369814,1,7147,73211
177401,999900,1524105,1,7147,4486
177402,999900,2185191,1,7147,83
177403,999900,2467371,1,7147,25210


In [18]:
from scipy.sparse import csr_matrix

R = csr_matrix(
    (
        user_item['interaction_weight'],
        (user_item['uidx'], user_item['item_idx'])
    ),
    shape=(len(user2id), len(item2id))
)

ItemKNN

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

item_sim = cosine_similarity(R.T)

In [20]:
np.fill_diagonal(item_sim, 0)

In [21]:
def recommend_itemknn(user_id, R, item_sim, user2id, id2item, k=10):
    if user_id not in user2id:
        return []

    u = user2id[user_id]

    user_vector = R[u]
    scores = user_vector.dot(item_sim)
    scores = np.asarray(scores).ravel()

    seen = user_vector.nonzero()[1]
    scores[seen] = -1

    top_k_idx = np.argsort(scores)[-k:][::-1]

    return [id2item[i] for i in top_k_idx]

In [22]:
uid = 922200

recs = recommend_itemknn(
    user_id=uid,
    R=R,
    item_sim=item_sim,
    user2id=user2id,
    id2item=id2item,
    k=10
)

print(recs)

[np.uint32(3238819), np.uint32(9269337), np.uint32(7262921), np.uint32(579861), np.uint32(1681417), np.uint32(5941561), np.uint32(5424655), np.uint32(5412781), np.uint32(2171386), np.uint32(2949604)]


In [23]:
test_true = test_clean.groupby('uid')['item_id'].apply(set).to_dict()

In [24]:
def recall_at_k(test_true, R, item_sim, user2id, id2item, k=10, recommend_fn=None):
    def default_recommend(uid, kk):
        return recommend_itemknn(uid, R, item_sim, user2id, id2item, kk)

    rec_fn = recommend_fn or default_recommend
    hits = 0
    total = 0

    for uid, true_items in test_true.items():
        recs = rec_fn(uid, k)
        recs_set = set(recs)
        hits += len(recs_set & true_items)
        total += len(true_items)

    return hits / total if total > 0 else 0

In [25]:
def precision_at_k(test_true, R, item_sim, user2id, id2item, k=10, recommend_fn=None):
    def default_recommend(uid, kk):
        return recommend_itemknn(uid, R, item_sim, user2id, id2item, kk)

    rec_fn = recommend_fn or default_recommend
    hits = 0
    users = 0

    for uid, true_items in test_true.items():
        recs = rec_fn(uid, k)
        if len(recs) == 0:
            continue
        hits += len(set(recs) & true_items)
        users += 1

    return hits / (users * k) if users > 0 else 0

In [26]:
def dcg_at_k(recs, true_items, k):
    dcg = 0.0

    for i, item in enumerate(recs[:k]):
        if item in true_items:
            dcg += 1 / np.log2(i + 2)

    return dcg


def idcg_at_k(true_items, k):
    ideal_hits = min(len(true_items), k)

    idcg = 0.0
    for i in range(ideal_hits):
        idcg += 1 / np.log2(i + 2)

    return idcg


def ndcg_at_k(test_true, R, item_sim, user2idx, id2item, k=10, recommend_fn=None):
    def default_recommend(uid, kk):
        return recommend_itemknn(uid, R, item_sim, user2idx, id2item, kk)

    rec_fn = recommend_fn or default_recommend
    ndcgs = []

    for uid, true_items in test_true.items():
        recs = rec_fn(uid, k)
        dcg = dcg_at_k(recs, true_items, k)
        idcg = idcg_at_k(true_items, k)
        if idcg == 0:
            continue
        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs) if ndcgs else 0

In [27]:
k = 10

recall = recall_at_k(test_true, R, item_sim, user2id, id2item, k)
precision = precision_at_k(test_true, R, item_sim, user2id, id2item, k)
ndcg = ndcg_at_k(test_true, R, item_sim, user2id, id2item, k)

print("NDCG@10:", ndcg)
print("Recall@10:", recall)
print("Precision@10:", precision)

NDCG@10: 0.0036148973876059414
Recall@10: 0.0030651340996168583
Precision@10: 0.0007670182166826462


UserKNN

In [28]:
user_sim = cosine_similarity(R)

In [29]:
np.fill_diagonal(user_sim, 0)

In [30]:
def recommend_userknn(user_id, R, user_sim, user2id, id2item, k=10):
    if user_id not in user2id:
        return []

    u = user2id[user_id]

    sim_vec = user_sim[u]
    # sim_vec @ R — корректное умножение (1, n_users) @ (n_users, n_items);
    # sim_vec.dot(R) с csr_matrix даёт вектор длины n_users — ошибка индексов.
    scores = np.asarray(sim_vec @ R).ravel()

    seen = R[u].nonzero()[1]
    scores[seen] = -1

    top_k_idx = np.argsort(scores)[-k:][::-1]

    return [id2item[i] for i in top_k_idx]

In [31]:
uid = 922200

recs = recommend_userknn(
    user_id=uid,
    R=R,
    user_sim=user_sim,
    user2id=user2id,
    id2item=id2item,
    k=10
)

print(recs)

[np.uint32(5252571), np.uint32(2185191), np.uint32(8647463), np.uint32(2408923), np.uint32(3542184), np.uint32(8823268), np.uint32(6006390), np.uint32(5436843), np.uint32(6813373), np.uint32(8200811)]


In [32]:
k = 10

recommend_user = lambda uid, kk: recommend_userknn(
    uid, R, user_sim, user2id, id2item, kk
)

recall = recall_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_user
)
precision = precision_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_user
)
ndcg = ndcg_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_user
)

print("NDCG@10 (UserKNN):", ndcg)
print("Recall@10 (UserKNN):", recall)
print("Precision@10 (UserKNN):", precision)

NDCG@10 (UserKNN): 0.017526777845615366
Recall@10 (UserKNN): 0.022222222222222223
Precision@10 (UserKNN): 0.005560882070949185


### iALS (implicit ALS)

Обучаем только на **лайках** (`weight > 0`): дизлайки не задают положительный implicit-сигнал. Библиотека [`implicit`](https://github.com/benfred/implicit).

Рекомендации считаются по обучающей матрице `R_pos` (лайки из трейна).

In [ ]:
import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

from implicit.als import AlternatingLeastSquares

coo = R.tocoo()
mask = coo.data > 0
R_pos = csr_matrix(
    (coo.data[mask].astype(np.float32), (coo.row[mask], coo.col[mask])),
    shape=R.shape,
)
R_pos.eliminate_zeros()

als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.08,
    iterations=25,
    random_state=42,
    num_threads=0,
)
als_model.fit(R_pos)


def recommend_ials(user_id, R_train, model, user2id, id2item, k=10):
    if user_id not in user2id:
        return []
    u = user2id[user_id]
    row = R_train[u]
    ids, _ = model.recommend(
        u, row, N=k, filter_already_liked_items=True
    )
    return [id2item[int(i)] for i in ids]

In [ ]:
k = 10

recommend_ials_fn = lambda uid, kk: recommend_ials(
    uid, R_pos, als_model, user2id, id2item, k=kk
)

recall = recall_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_ials_fn
)
precision = precision_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_ials_fn
)
ndcg = ndcg_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_ials_fn
)

print("NDCG@10 (iALS):", ndcg)
print("Recall@10 (iALS):", recall)
print("Precision@10 (iALS):", precision)

### EASE

Полная матрица $X^\top X$ по всем айтемам здесь заняла бы порядка десятков ГБ RAM, поэтому ниже — **EASE на топ-K популярных айтемов** (по сумме весов в `R_pos`). Кандидаты для ранжирования только из этого поднабора; метрики сопоставимы с KNN/iALS только качественно.

Параметры `EASE_TOPK` и `EASE_LAM` можно подкрутить.

In [ ]:
EASE_TOPK = 5000
EASE_LAM = 250.0

col_pop = np.array(R_pos.sum(axis=0)).ravel()
K_ease = min(EASE_TOPK, R_pos.shape[1])
top_part = np.argpartition(-col_pop, K_ease - 1)[:K_ease]
ease_item_cols = top_part[np.argsort(-col_pop[top_part])].astype(np.int64)

X_bin = (R_pos[:, ease_item_cols].toarray() > 0).astype(np.float64)
G = X_bin.T @ X_bin
B_ease = np.linalg.solve(G + EASE_LAM * np.eye(G.shape[0]), G)
np.fill_diagonal(B_ease, 0.0)


def recommend_ease(user_id, R_train, B, ease_cols, user2id, id2item, k=10):
    if user_id not in user2id:
        return []
    u = user2id[user_id]
    row = R_train[u].toarray().ravel()
    x = row[ease_cols].astype(np.float64)
    x_bin = (x > 0).astype(np.float64)
    scores = x_bin @ B
    seen = np.flatnonzero(x_bin)
    scores[seen] = -1.0
    n_take = min(k, scores.shape[0])
    top_local = np.argsort(scores)[-n_take:][::-1]
    return [id2item[int(ease_cols[j])] for j in top_local]

In [ ]:
k = 10

recommend_ease_fn = lambda uid, kk: recommend_ease(
    uid, R_pos, B_ease, ease_item_cols, user2id, id2item, k=kk
)

recall = recall_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_ease_fn
)
precision = precision_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_ease_fn
)
ndcg = ndcg_at_k(
    test_true, R, item_sim, user2id, id2item, k, recommend_fn=recommend_ease_fn
)

print(f"NDCG@10 (EASE, top-{len(ease_item_cols)} items):", ndcg)
print("Recall@10 (EASE):", recall)
print("Precision@10 (EASE):", precision)